# 🎓 Machine Learning Project: Personalized AI Tutor via LLM & Reinforcement Learning
## 📌 Step 04: Feature Engineering & 1-PL Item Response Theory (IRT)

---

### 📖 Objective:
1. Engineer cumulative student history features (cumulative attempts, historical accuracy).
2. Fit the **1-Parameter Logistic (1-PL / Rasch) Item Response Theory Model**:
   $$P(\text{correct} \mid \theta, b_i) = \frac{1}{1 + e^{-(\theta - b_i)}}$$
3. Discretize skill difficulties into 5 pedagogical challenge tiers (Levels 1 to 5).
4. Plot **Item Characteristic Curves (ICC)** across latent ability $\theta \in [-3, 3]$.


### Cell 1: Import Libraries


In [ ]:
# Cell 1: Imports
%matplotlib inline
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("✅ Libraries Imported Successfully")


### Cell 2: Load Preprocessed Dataset


In [ ]:
# Cell 2: Load Data
cur_dir = os.path.abspath(".")
DATA_DIR = os.path.join(cur_dir, "data") if os.path.exists("data") else cur_dir
PROCESSED_PATH = os.path.join(DATA_DIR, "assistments_processed.csv")
FEAT_PATH = os.path.join(DATA_DIR, "assistments_features.csv")

if not os.path.exists(PROCESSED_PATH):
    RAW_PATH = os.path.join(DATA_DIR, "skill_builder_data_corrected.csv")
    df = pd.read_csv(RAW_PATH, low_memory=False, encoding='latin1')
else:
    df = pd.read_csv(PROCESSED_PATH)

print(f"Loaded {len(df):,} interaction rows.")
print("✅ Dataset Loaded Successfully")


### Cell 3: Compute Cumulative Historical Features


In [ ]:
# Cell 3: Cumulative Student Features
df["cumulative_attempts"] = df.groupby(["user_id", "skill_name"]).cumcount() + 1
df["cum_correct"] = df.groupby(["user_id", "skill_name"])["correct"].cumsum() - df["correct"]
df["prior_accuracy"] = (df["cum_correct"] / np.maximum(1, df["cumulative_attempts"] - 1)).round(3)
df.loc[df["cumulative_attempts"] == 1, "prior_accuracy"] = 0.50

print("--- Sample Feature-Engineered Rows: ---")
display(df[["user_id", "skill_name", "correct", "cumulative_attempts", "prior_accuracy"]].head(6))


### Cell 4: 1-PL IRT Difficulty Calibration & 5-Tier Discretization


In [ ]:
# Cell 4: Estimate IRT Difficulty Parameters
skill_agg = df.groupby("skill_name")["correct"].agg(["count", "mean"]).reset_index()
skill_agg.columns = ["skill_name", "total_interactions", "empirical_p_correct"]

# 1-PL IRT formula: b_i = ln((1 - p) / p)
p_safe = skill_agg["empirical_p_correct"].clip(1e-4, 1.0 - 1e-4)
skill_agg["irt_difficulty_b"] = np.log((1.0 - p_safe) / p_safe).round(4)

# Discretize into 5 Difficulty Tiers
b_min, b_max = skill_agg["irt_difficulty_b"].min(), skill_agg["irt_difficulty_b"].max()
if b_max > b_min:
    skill_agg["difficulty_tier"] = 1 + np.round(4.0 * (skill_agg["irt_difficulty_b"] - b_min) / (b_max - b_min)).astype(int)
else:
    skill_agg["difficulty_tier"] = 3

df_feat = df.merge(skill_agg[["skill_name", "irt_difficulty_b", "difficulty_tier"]], on="skill_name", how="left")
df_feat.to_csv(FEAT_PATH, index=False)

print("--- Calibrated Skill Difficulty Table: ---")
display(skill_agg)


### Cell 5: Item Characteristic Curves (ICC) Visualization


In [ ]:
# Cell 5: Plot ICC Curves
theta_range = np.linspace(-3.0, 3.0, 100)
plt.figure(figsize=(11, 5.5))

for _, row in skill_agg.head(8).iterrows():
    b = row["irt_difficulty_b"]
    p_theta = 1.0 / (1.0 + np.exp(-(theta_range - b)))
    plt.plot(theta_range, p_theta, lw=2.2, label=f"{row['skill_name'][:24]} (b={b:.2f})")

plt.axhline(0.5, color='gray', linestyle='--', alpha=0.6, label="50% Threshold")
plt.title("1-PL Item Response Theory (Rasch) - Item Characteristic Curves (ICC)", fontsize=13, fontweight="bold")
plt.xlabel("Student Latent Ability (θ)", fontsize=11)
plt.ylabel("Probability of Correct Response P(θ)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()
print("✅ Step 04 Completed Successfully!")
